# Project 5: Self-Healing Code & Debugger Agent (The ReAct Sandbox Loop)

This notebook implements an enterprise-grade **Self-Healing Code Generation & Debugging Pipeline** featuring:
- **LangGraph StateGraph**: Deterministic state passing across node boundaries (`CodeDebugState`).
- **Specialized Multi-Agent Nodes**:
  1. **Coder Agent (Generator/Reflector)**: Writes initial code and applies patch fixes based on compiler errors.
  2. **Security Guard Agent (AST Validator)**: Parses code AST to block unsafe imports (`os`, `subprocess`, `shutil`) and dangerous calls (`eval`, `exec`).
  3. **Sandbox Executor**: Executes code in an isolated subprocess with a 5-second execution timeout.
  4. **Debugger Agent (Traceback Analyzer)**: Intercepts `stderr` tracebacks (`IndexError`, `ZeroDivisionError`, `AssertionError`) and generates targeted repair instructions.
- **Direct Reflection Repair Loop**: Automated feedback loop (`debugger` ──► `coder`) with a `max_revisions = 3` safety cap.


In [1]:
import sys
import os

# Add project root to sys.path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Environment configured!")



Environment configured!


In [2]:
# 1. Test AST Security Validator
from src.guardrails.code_validator import validate_python_code

clean_code = "def add(a, b): return a + b\nassert add(2, 3) == 5"
unsafe_code = "import os\nos.system('echo hacked')"

print("Checking clean code:", validate_python_code(clean_code))
print("Checking unsafe code:", validate_python_code(unsafe_code))



Checking clean code: {'is_valid': True, 'error_type': None, 'message': 'Code AST validation passed cleanly.'}
Checking unsafe code: {'is_valid': False, 'error_type': 'SECURITY_VIOLATION', 'message': "Security policy violations found: Forbidden import 'os' detected.; Forbidden attribute call '.system()' detected."}


In [3]:
# 2. Test Isolated Subprocess Sandbox
from src.tools.sandbox_executor import execute_code_in_sandbox

buggy_snippet = "x = 10 / 0"
print("Executing buggy snippet in sandbox...")
res = execute_code_in_sandbox(buggy_snippet)
print("Success:", res["success"])
print("Status:", res["status"])
print("Stderr Captured:\n", res["stderr"])



Executing buggy snippet in sandbox...
Success: False
Status: RUNTIME_ERROR
Stderr Captured:
 Traceback (most recent call last):
  File "/var/folders/q6/y1570qd146q_h_cy44rd1qfr0000gn/T/tmp5iulb7ja.py", line 1, in <module>
    x = 10 / 0
        ~~~^~~
ZeroDivisionError: division by zero



In [4]:
# 3. Initialize Self-Healing Debugger Agent Graph
from src.agents.debugger_agent import init_debugger_agent

print("🛠️ Initializing Self-Healing StateGraph Agent...")
agent = init_debugger_agent()
print("✅ Agent compiled successfully!")



🛠️ Initializing Self-Healing StateGraph Agent...
✅ Agent compiled successfully!


In [5]:
# 4. Execute End-to-End Self-Healing Test
task = "Write a Python function `parse_even_numbers(lst)` that filters even numbers from a list of strings or ints, handling mixed types. Include assertion test cases."

print(f"🚀 Running Self-Healing Pipeline for task: '{task}'\n" + "="*70)

initial_state = {"task_description": task, "revision_count": 0}
final_state = agent.invoke(initial_state)

print("\n" + "="*70)
print(f"🎉 PIPELINE COMPLETE! Status: '{final_state.get('status')}'")
print(f"Revisions Required: {final_state.get('revision_count', 0)}")
print("="*70 + "\nFINAL VERIFIED CODE:\n")
print(final_state.get("code"))



🚀 Running Self-Healing Pipeline for task: 'Write a Python function `parse_even_numbers(lst)` that filters even numbers from a list of strings or ints, handling mixed types. Include assertion test cases.'
👨‍💻 [CODER AGENT - INITIAL GENERATION MODE] Writing solution for task: 'Write a Python function `parse_even_numbers(lst)` that filters even numbers from a list of strings or ints, handling mixed types. Include assertion test cases.'...
✅ [CODER AGENT] Code draft generated successfully!
🛡️ [SECURITY GUARD] Auditing code AST for syntax & security policies...
✅ [SECURITY GUARD] AST validation passed cleanly!
🏃 [SANDBOX EXECUTOR] Running code in isolated subprocess...
🎉 [SANDBOX EXECUTOR] Code executed and passed all assertions!
🏁 [SELF-HEALING COMPLETE] Code execution passed all tests! Routing to END.

🎉 PIPELINE COMPLETE! Status: 'SUCCESS'
Revisions Required: 0
FINAL VERIFIED CODE:

def parse_even_numbers(lst):
    result = []
    for item in lst:
        try:
            num = int(item)